<a href="https://colab.research.google.com/github/Abdulrhmandarwish/the-Starter-Notebooks/blob/main/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")
print("HF_TOKEN found:", bool(token))

HF_TOKEN found: True


In [2]:
import pandas as pd

## 1. Unit of analysis + time window

**One row means:** in `fact_content_daily_performance`, one row = one client's one
content item's performance on one calendar day (grain: report_date × client_hash_id ×
content_hash_id).

**Table(s) I'll use:** `fact_content_daily_performance` (primary), joined to
`dim_content` on `content_hash_id` for static metadata (word_count, content_created_at),
and to `dim_clients` on `client_hash_id` only to read `ga4_data_start` — never to pull
features from.

**Time window:** developing on the `month=2026-03` partition (a mid-panel month,
report_date 2026-03-01 → 2026-03-31). The final month (2026-06, the `_sample` table)
stays sealed as test — I don't touch it here.

**What I'd predict/rank:** `ctr_underperform` — a proxy label, 1 when a row's realized
CTR falls below the expected CTR for its position bucket, else 0. Same caveat as the
starter's `trend_direction`: this is a rule-derived proxy from today's numbers, not a
future observed outcome.

**Deliberately excluded:** raw `gsc_clicks` and the CTR ratio itself — they're what the
label is computed from, so they never become features. `client_hash_id` /
`content_hash_id` — join/grouping keys only. GA4 columns on rows where
`ga4_data_available = FALSE` — those are zero-filled ("not tracked yet"), not real zeros.

In [3]:
%pip -q install duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

| Bucket   | Fields | Why |
|---|---|---|
| Feature  | `gsc_avg_position`, `gsc_impressions`, `content_age_days` (derived), `ga4_engaged_sessions`/`engagement_rate` (only where `ga4_data_available IS TRUE`), `word_count` | each is already logged/fixed as of report_date |
| Label/proxy | `ctr_underperform` (built from `gsc_clicks`, `gsc_impressions` → ctr, vs. position-bucket expected CTR) | the thing I'm ranking on |
| Context  | `report_date`, `client_hash_id`, `content_hash_id`, `month` partition | join/group/filter only, never learned from |
| Excluded | `gsc_clicks`, raw ctr ratio — feeds the label directly; GA4 rows with `ga4_data_available = FALSE` — zero-filled, not real zero engagement | leakage / not-yet-tracked |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# ============================================================
# W03 — Verification Queries + Five Features + Leakage Trap
# ============================================================

# ---- Query 1: grain check ----
grain = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS c
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print("duplicate grain rows (should be empty):")
print(grain)


# ---- Query 2: slice row count + date span ----
span = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content
    FROM {FACT}
""").df()

print(span)


# ---- Query 3: availability, filtered with IS TRUE ----
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE THEN 1
                ELSE 0
            END
        ) AS available_rows
    FROM {FACT}
""").df()

print(avail)

survivors = con.sql(f"""
    SELECT COUNT(*) AS n_survivors
    FROM {FACT}
    WHERE ga4_data_available IS TRUE
""").df()

print(survivors)


# ============================================================
# Five features
# ============================================================

features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,

        -- Five honest features
        f.gsc_avg_position,
        f.gsc_impressions,
        DATE_DIFF(
            'day',
            c.content_created_date,
            f.report_date
        ) AS content_age_days,
        f.ga4_engaged_sessions,
        c.word_count,

        -- Label ingredients kept separate
        f.gsc_clicks,
        f.gsc_impressions AS impr_for_ctr

    FROM {FACT} AS f

    JOIN {DIM_CONTENT} AS c
        ON f.content_hash_id = c.content_hash_id

    WHERE f.ga4_data_available IS TRUE
""").df()

print("Feature frame shape:", features.shape)
print(features.head())


# ============================================================
# The trap: deliberately create label leakage
# ============================================================

# Build CTR
features["ctr"] = (
    features["gsc_clicks"]
    / features["impr_for_ctr"].replace(0, pd.NA)
)


# Expected CTR within each search-position bucket
pos_bucket_expected = (
    features
    .groupby(
        pd.cut(
            features["gsc_avg_position"],
            [0, 3, 10, 20, 100]
        )
    )["ctr"]
    .transform("median")
)


# Create the label
features["ctr_underperform"] = (
    features["ctr"] < pos_bucket_expected
).astype(int)


# ============================================================
# Train/test split
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


# ---- Honest model: five features only ----
honest_X = features[
    [
        "gsc_avg_position",
        "gsc_impressions",
        "content_age_days",
        "ga4_engaged_sessions",
        "word_count"
    ]
].fillna(0)

y = features["ctr_underperform"]


Xtr, Xte, ytr, yte = train_test_split(
    honest_X,
    y,
    test_size=0.3,
    random_state=0
)


honest_model = LogisticRegression(
    max_iter=1000
)

honest_model.fit(Xtr, ytr)

honest_auc = roc_auc_score(
    yte,
    honest_model.predict_proba(Xte)[:, 1]
)

print("HONEST auc (5 features only):", honest_auc)


# ============================================================
# Deliberate leakage
# ============================================================

leaky_X = honest_X.copy()

# LEAK:
# CTR was directly used to construct the label.
leaky_X["ctr"] = features["ctr"].fillna(0)


Xtr, Xte, ytr, yte = train_test_split(
    leaky_X,
    y,
    test_size=0.3,
    random_state=0
)


leaky_model = LogisticRegression(
    max_iter=1000
)

leaky_model.fit(Xtr, ytr)

leaky_auc = roc_auc_score(
    yte,
    leaky_model.predict_proba(Xte)[:, 1]
)

print(
    "LEAKY auc (with ctr column):",
    leaky_auc,
    "<- should jump toward 1.0"
)


# ============================================================
# Delete the leak and keep the honest score
# ============================================================

del leaky_X

print("Kept score: honest_auc =", honest_auc)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate grain rows (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
    n_rows   min_date   max_date  n_clients  n_content
0  9841378 2026-03-01 2026-03-31         55     331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  available_rows
0     9841378        413966.0
   n_survivors
0       413966


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (413966, 10)
            client_hash_id           content_hash_id report_date  \
0  client_9958f0a7ae1df715  content_810cf06597918291  2026-03-01   
1  client_9958f0a7ae1df715  content_eb0aeedbcfaf2712  2026-03-01   
2  client_9958f0a7ae1df715  content_b813c73d7000b3b1  2026-03-01   
3  client_9958f0a7ae1df715  content_651b8ba180f9beff  2026-03-01   
4  client_9958f0a7ae1df715  content_1f39e904c7351258  2026-03-01   

   gsc_avg_position  gsc_impressions  content_age_days  ga4_engaged_sessions  \
0         11.272727               11               338                     1   
1          5.367347               49               338                     0   
2          5.642857               14               338                     0   
3          1.600000                5               338                     0   
4          7.214286               28               338                     0   

   word_count  gsc_clicks  impr_for_ctr  
0        2946           0         

/tmp/ipykernel_694/3616202819.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(


HONEST auc (5 features only): 0.7262387539031294


/tmp/ipykernel_694/3616202819.py:180: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  leaky_X["ctr"] = features["ctr"].fillna(0)


LEAKY auc (with ctr column): 0.8338806495645914 <- should jump toward 1.0
Kept score: honest_auc = 0.7262387539031294


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

**Unbalanced panel.** Client history depth varies wildly (`gsc_data_start` differs by
client), so `month=2026-03` mixes clients with a full year of history against clients
only weeks in. Averaging or ranking across clients in this slice without a per-client
baseline conflates "thin new tracking" with "genuinely weak performance" — this data
can't tell those two apart without client-level normalization first.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.